# PILOT behavior cloning: Qwen2.5-7B-Instruct LoRA on the randomized-rollout corpus

Trains the Stage-1 BC orchestrator (PILOT §4.6) on `data/bc/oai_train.jsonl` /
`oai_dev.jsonl` produced by `adr bc-pairs`, checks it on the held-out dev
queries, saves the adapter to Google Drive, and serves it as an
OpenAI-compatible endpoint (through a Cloudflare tunnel) so the fork's
`GR_ORCHESTRATOR=learned` policy can call it from your laptop.

**Runtime:** Runtime → Change runtime type → GPU. A100 / L4 are best; T4 (16 GB)
works for training (QLoRA) and for the FastAPI server, but not for vLLM.

**Sections**
1. Setup + data
2. Train (QLoRA)
3. Dev check (format validity, KEEP Jaccard, DECISION accuracy)
4. Save adapter to Drive
5. Serve (FastAPI or vLLM) + tunnel → copy the URL to the laptop

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%pip install -q -U "transformers>=4.46" "peft>=0.13" "trl>=0.21" "accelerate>=1.0" "bitsandbytes>=0.44" "datasets>=3.0" fastapi uvicorn nest_asyncio
import torch, transformers, trl, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| trl", trl.__version__, "| peft", peft.__version__)
print("bf16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
# Mount Drive: data goes in  MyDrive/pilot_bc/data/ , the adapter is saved to  MyDrive/pilot_bc/adapters/
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
ROOT = Path('/content/drive/MyDrive/pilot_bc')
DATA = ROOT / 'data'
ADAPTERS = ROOT / 'adapters'
DATA.mkdir(parents=True, exist_ok=True); ADAPTERS.mkdir(parents=True, exist_ok=True)

TRAIN = DATA / 'oai_train.jsonl'
DEV = DATA / 'oai_dev.jsonl'
if not TRAIN.exists():
    # Fallback: upload the two files from your laptop (deepresearchagent/data/bc/oai_*.jsonl)
    from google.colab import files
    up = files.upload()
    for name, blob in up.items():
        (DATA / name).write_bytes(blob)
print("train:", TRAIN.exists(), "dev:", DEV.exists())

In [ ]:
import json
from datasets import Dataset

def load_pc(path):
    # OpenAI chat rows -> TRL conversational prompt/completion (loss on the assistant turn only)
    rows = []
    for line in open(path):
        m = json.loads(line)["messages"]
        assert m[-1]["role"] == "assistant"
        rows.append({"prompt": m[:-1], "completion": [m[-1]]})
    return Dataset.from_list(rows)

train_ds = load_pc(TRAIN)
dev_ds = load_pc(DEV)
print(train_ds, dev_ds)
print(train_ds[0]["completion"][0]["content"])

## 2. Train (QLoRA, 4-bit base, LoRA r=16 on all projections)

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

BASE = "Qwen/Qwen2.5-7B-Instruct"
RUN_NAME = "qwen25-7b-pilot-bc-v1"
OUT = f"/content/out/{RUN_NAME}"
BF16 = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16 else torch.float16

tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.pad_token or tok.eos_token

def n_tokens(ex):
    out = tok.apply_chat_template(ex["prompt"] + ex["completion"], tokenize=True, add_generation_prompt=False)
    ids = out["input_ids"] if not isinstance(out, list) else out
    if ids and isinstance(ids[0], (list, tuple)):
        ids = ids[0]
    return len(ids)
# Safety cap only. The fused-kernel patch below is what keeps a T4 from
# materialising the heads*seq*seq score matrix. Truncation would cut the
# completion, which is the label, so over-long rows are dropped instead.
CAP = 8192
train_ds = train_ds.filter(lambda ex: n_tokens(ex) <= CAP)
dev_ds = dev_ds.filter(lambda ex: n_tokens(ex) <= CAP)
lens = [n_tokens(ex) for ex in train_ds]
print(f"kept train {len(train_ds)} dev {len(dev_ds)} | tokens mean {sum(lens)/len(lens):.0f} max {max(lens)}")
MAX_LEN = CAP

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, torch_dtype=DTYPE,
                                             device_map={"": 0}, attn_implementation="sdpa")
model.config.use_cache = False

# The dense causal mask makes SDPA take the math kernel, which allocates
# heads*seq*seq (~1.7 GiB at 4096, ~4 GiB at 6k) and OOMs a 16 GB T4.
# Batch size is 1 and nothing is padded, so drop the mask and use the fused
# causal kernel. Math is disabled so a failed patch errors instead of OOMing.
import sys
import transformers.integrations.sdpa_attention as _sdpa
_orig_sdpa = _sdpa.sdpa_attention_forward
# T4 mem-efficient attention cannot do grouped-query (enable_gqa). Force the
# repeat-kv path, and drop the dense mask so the fused causal kernel is used
# instead of the math kernel that OOMs.
_sdpa.use_gqa_in_sdpa = lambda *args, **kwargs: False

def _causal_sdpa(module, query, key, value, attention_mask=None, dropout=0.0, scaling=None, is_causal=None, **kwargs):
    if query.shape[0] == 1:
        attention_mask = None
        kwargs.pop("position_bias", None)
        is_causal = True
    return _orig_sdpa(module, query, key, value, attention_mask=attention_mask, dropout=dropout, scaling=scaling, is_causal=is_causal, **kwargs)

def _install_sdpa(fn):
    hit = 0
    for _mod in list(sys.modules.values()):
        if getattr(_mod, "sdpa_attention_forward", None) is _orig_sdpa:
            setattr(_mod, "sdpa_attention_forward", fn); hit += 1
        iface = getattr(_mod, "ALL_ATTENTION_FUNCTIONS", None)
        mapping = getattr(iface, "_global_mapping", None) if iface is not None else None
        if isinstance(mapping, dict) and mapping.get("sdpa") is _orig_sdpa:
            mapping["sdpa"] = fn; hit += 1
    return hit
print("sdpa patch sites", _install_sdpa(_causal_sdpa))
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(False)

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])

cfg = SFTConfig(
    output_dir=OUT,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=2,            # ~5% of ~39 steps; current TRL dropped warmup_ratio
    weight_decay=0.0,
    logging_steps=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    bf16=BF16, fp16=not BF16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=MAX_LEN,
    packing=False,
    completion_only_loss=True,   # loss on KEEP/ALLOC/DECISION only
    report_to="none",
    seed=17,
)
trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds, eval_dataset=dev_ds,
                     processing_class=tok, peft_config=lora)
trainer.train()
print(trainer.state.log_history[-3:])

## 3. Dev check: does it emit valid actions, and how close to the cloned ones?

In [ ]:
import re, torch

_KEEP = re.compile(r"^\s*KEEP\s*:\s*(.*)$", re.I | re.M)
_ALLOC = re.compile(r"^\s*ALLOC\s*:\s*(.*)$", re.I | re.M)
_DEC = re.compile(r"^\s*DECISION\s*:\s*(CONTINUE|TERMINATE)", re.I | re.M)
_PAIR = re.compile(r"([A-Za-z0-9_\-]+)\s*=\s*([0-9]*\.?[0-9]+)")

def parse(text):
    k = _KEEP.search(text or ""); a = _ALLOC.search(text or ""); d = _DEC.search(text or "")
    keep = None
    if k:
        body = k.group(1).strip()
        keep = "ALL" if body.upper() == "ALL" else set(re.findall(r"[A-Za-z0-9_\-]+", body))
    alloc = {n: float(w) for n, w in _PAIR.findall(a.group(1))} if a else {}
    s = sum(alloc.values()) or 1.0
    alloc = {n: w / s for n, w in alloc.items()}
    dec = d.group(1).upper() if d else None
    return keep, alloc, dec

def pool_ids(prompt_text):
    ids = []
    grab = False
    for line in prompt_text.splitlines():
        if line.startswith("Evidence ("): grab = True; continue
        if grab:
            if not line.strip(): break
            ids.append(line.strip().split(" | ")[0])
    return ids

def encode_chat(messages):
    # Newer tokenizers return a BatchEncoding dict; older ones a bare tensor.
    out = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
    ids = out["input_ids"] if not isinstance(out, torch.Tensor) else out
    return ids.to(model.device)

@torch.no_grad()
def generate(messages, max_new_tokens=1200):
    # Each 12-char evidence id is ~9 Qwen tokens; gold completions run up to ~930.
    ids = encode_chat(messages)
    out = model.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

model.eval()
model.config.use_cache = True
rows = []
for ex in dev_ds:
    pred = generate(ex["prompt"])
    gold = ex["completion"][0]["content"]
    pk, pa, pd_ = parse(pred); gk, ga, gd = parse(gold)
    pool = set(pool_ids(ex["prompt"][-1]["content"]))
    pk = pool if pk == "ALL" else (pk or set()); gk = pool if gk == "ALL" else (gk or set())
    jacc = len(pk & gk) / len(pk | gk) if (pk | gk) else 1.0
    l1 = sum(abs(pa.get(n, 0) - ga.get(n, 0)) for n in set(pa) | set(ga))
    rows.append(dict(valid=bool(pk) and pd_ is not None, keep_jaccard=round(jacc, 3), keep_frac_pred=round(len(pk)/max(1,len(pool)),2),
                     keep_frac_gold=round(len(gk)/max(1,len(pool)),2), alloc_l1=round(l1, 3), dec_pred=pd_, dec_gold=gd, dec_ok=pd_ == gd,
                     hallucinated_ids=len(pk - pool)))
import pandas as pd
df = pd.DataFrame(rows); display(df)
print("valid %.2f | KEEP Jaccard %.3f | DECISION acc %.2f | ALLOC L1 %.3f | hallucinated ids %d" %
      (df.valid.mean(), df.keep_jaccard.mean(), df.dec_ok.mean(), df.alloc_l1.mean(), df.hallucinated_ids.sum()))
print("\n--- sample ---\n", pred)

## 4. Save the adapter to Drive

In [ ]:
SAVE = ADAPTERS / RUN_NAME
trainer.model.save_pretrained(str(SAVE))
tok.save_pretrained(str(SAVE))
(SAVE / "train_log.json").write_text(json.dumps(trainer.state.log_history, indent=1))
print("saved:", sorted(p.name for p in SAVE.iterdir()))

## 5. Serve it for the laptop

Two options. **5a (FastAPI)** runs on any GPU incl. T4 and reuses the model already
in memory. **5b (vLLM)** is much faster but needs ≥24 GB (L4/A100) and a fresh
`pip install vllm` (~5 min). Either way the last cell prints the `GR_ORCH_LLM_*`
exports to paste on the laptop.

If you restart the runtime, run the *Reload adapter* cell instead of training again.

In [ ]:
# (Only if the runtime was restarted) reload base + adapter from Drive
if "model" not in globals():
    import torch, json
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import PeftModel
    from pathlib import Path
    BASE = "Qwen/Qwen2.5-7B-Instruct"; RUN_NAME = "qwen25-7b-pilot-bc-v1"
    ADAPTERS = Path('/content/drive/MyDrive/pilot_bc/adapters'); SAVE = ADAPTERS / RUN_NAME
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    tok = AutoTokenizer.from_pretrained(str(SAVE))
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True)
    base = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, torch_dtype=DTYPE, device_map={"": 0})
    model = PeftModel.from_pretrained(base, str(SAVE)).eval()
    print("reloaded", SAVE)

In [ ]:
# 5a. Minimal OpenAI-compatible server (chat.completions) over the in-memory model
import time, uuid, threading, asyncio, torch
import nest_asyncio, uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List, Optional

nest_asyncio.apply()
app = FastAPI()
_lock = threading.Lock()
MODEL_ID = "pilot-bc"

class Msg(BaseModel):
    role: str
    content: str

class ChatReq(BaseModel):
    model: str = MODEL_ID
    messages: List[Msg]
    temperature: Optional[float] = 0.0
    max_tokens: Optional[int] = 1200
    stream: Optional[bool] = False

@app.get("/v1/models")
def models():
    return {"object": "list", "data": [{"id": MODEL_ID, "object": "model", "owned_by": "pilot"}]}

@app.post("/v1/chat/completions")
def chat(req: ChatReq):
    msgs = [m.model_dump() for m in req.messages]
    with _lock, torch.no_grad():
        enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt")
        ids = (enc["input_ids"] if not isinstance(enc, torch.Tensor) else enc).to(model.device)
        do_sample = bool(req.temperature and req.temperature > 0)
        out = model.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=req.max_tokens or 1200, do_sample=do_sample,
                             temperature=req.temperature if do_sample else None, pad_token_id=tok.pad_token_id)
    text = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
    return {
        "id": f"chatcmpl-{uuid.uuid4().hex[:12]}", "object": "chat.completion", "created": int(time.time()), "model": MODEL_ID,
        "choices": [{"index": 0, "message": {"role": "assistant", "content": text}, "finish_reason": "stop"}],
        "usage": {"prompt_tokens": int(ids.shape[1]), "completion_tokens": int(out.shape[1] - ids.shape[1]),
                  "total_tokens": int(out.shape[1])},
    }

PORT = 8000
server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(2)
import requests
print(requests.get(f"http://127.0.0.1:{PORT}/v1/models").json())

In [ ]:
# 5b. (optional, L4/A100 only) vLLM with the LoRA adapter — faster; skip 5a if you use this.
# %pip install -q vllm
# import subprocess, time
# PORT = 8000
# vllm_proc = subprocess.Popen(["vllm", "serve", "Qwen/Qwen2.5-7B-Instruct", "--enable-lora",
#     "--lora-modules", f"pilot-bc={SAVE}", "--max-model-len", "12288", "--max-lora-rank", "16",
#     "--dtype", "bfloat16", "--port", str(PORT), "--served-model-name", "Qwen/Qwen2.5-7B-Instruct"],
#     stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT)
# for _ in range(120):
#     time.sleep(5)
#     try:
#         import requests; requests.get(f"http://127.0.0.1:{PORT}/v1/models").raise_for_status(); break
#     except Exception: pass
# print(open("/content/vllm.log").read()[-1500:])

In [ ]:
# Cloudflare quick tunnel -> public https URL for the laptop
import subprocess, re, time, os
if not os.path.exists("/content/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
t0 = time.time()
while time.time() - t0 < 60 and url is None:
    line = tunnel.stdout.readline()
    m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
    if m: url = m.group(0)
assert url, "tunnel did not come up; re-run this cell"

import requests
r = requests.post(f"{url}/v1/chat/completions", json={"model": "pilot-bc", "messages": dev_ds[0]["prompt"] if "dev_ds" in globals() else [{"role":"user","content":"KEEP: ALL\nALLOC: -\nDECISION: CONTINUE"}], "max_tokens": 300}, timeout=300)
print("status", r.status_code); print(r.json()["choices"][0]["message"]["content"][:400])

print("\n=== paste on the laptop (deepresearchagent/) ===")
print(f"export GR_ORCH_LLM_BASE_URL={url}/v1")
print("export GR_ORCH_LLM_MODEL=pilot-bc")
print("export GR_ORCH_LLM_API_KEY=EMPTY")
print("Keep this notebook running while the eval runs; the URL dies with the runtime.")